# 02 Preprocessing and Feature Engineering

This notebook prepares the first customer-level feature table for future segmentation. The table uses `customer_info` as the base because it contains the full customer population. Basket-derived features are added with a left join so customers without sampled baskets remain in the table.

This phase does not train clustering models and does not create final clustering outputs.

In [2]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)


def has_raw_datasets(candidate):
    root_layout = (candidate / "customer_info.csv").exists() and (candidate / "customer_basket.csv").exists()
    project_files_layout = (candidate / "Project files" / "customer_info.csv").exists() and (candidate / "Project files" / "customer_basket.csv").exists()
    return root_layout or project_files_layout


def find_project_root(start):
    for candidate in [start, *start.parents]:
        if has_raw_datasets(candidate):
            return candidate
    raise FileNotFoundError("Could not locate raw datasets at the repository root or in Project files/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REFERENCE_DATE = pd.Timestamp("2026-05-30")
print(f"Project root: {PROJECT_ROOT}")
print(f"Reference date for age and tenure: {REFERENCE_DATE.date()}")

Project root: C:\Users\pedro\Desktop\ML-II-projeto-final
Reference date for age and tenure: 2026-05-31


In [3]:
from src.data_loading import load_datasets
from src.data_audit import missing_values
from src.features import build_customer_feature_table

## Feature Engineering Decisions

- `customer_info` is the base table, so the feature table must preserve every customer.
- `customer_id` is retained only as the row key. Raw identifiers such as `customer_name` and `loyalty_card_number` are not modeling features.
- A fixed reference date (`2026-05-30`) is used so `customer_age` and `customer_tenure_years` are reproducible.
- Birthdate is converted to `customer_age`; invalid or missing ages are flagged before numeric imputation.
- First transaction year is converted to `customer_tenure_years`; future or implausible years are flagged and handled before imputation.
- Loyalty card number is converted to `has_loyalty_card` and `loyalty_card_missing`.
- Promotion percentages are clipped to `[0, 1]` for `promotion_pct_clean`, with suspicious values preserved in a flag.
- Missing numeric customer values are median-imputed with missingness indicators when a column has missing values.
- Basket features are computed per customer and left-joined to the full customer base. Customers without sampled baskets receive zero defaults for basket counts and sizes.

## Load Raw Data

In [4]:
customer_info, customer_basket = load_datasets(PROJECT_ROOT)

customers_before = customer_info["customer_id"].nunique()
print(f"customer_info rows: {len(customer_info):,}")
print(f"unique customers before feature engineering: {customers_before:,}")
print(f"customer_basket rows: {len(customer_basket):,}")

customer_info rows: 33,038
unique customers before feature engineering: 33,038
customer_basket rows: 100,000


## Build Customer-Level Feature Table

In [5]:
feature_table, metadata = build_customer_feature_table(
    customer_info,
    customer_basket,
    reference_date=REFERENCE_DATE,
)

customers_after = feature_table["customer_id"].nunique()
print(f"feature table shape: {feature_table.shape}")
print(f"unique customers after feature engineering: {customers_after:,}")
print(f"basket parse errors: {metadata['basket_parse_errors']:,}")
print(f"customers without sampled baskets: {metadata['customers_without_baskets']:,}")
display(feature_table.head())

feature table shape: (33038, 77)
unique customers after feature engineering: 33,038
basket parse errors: 0
customers without sampled baskets: 4,911


,customer_id,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,typical_hour,lifetime_spend_vegetables,lifetime_spend_nonalcohol_drinks,...,has_kids_home,has_teens_home,has_children_home,basket_count,avg_basket_size,median_basket_size,max_basket_size,total_basket_items,unique_basket_products,has_sampled_basket
0,3,1.0,1.0,1.0,3.0,11731.0,4553.0,12.0,373.0,323.0,...,1,1,1,2,10.5,10.5,11,21,21,1
1,4,1.0,0.0,0.0,2.0,13694.0,963.0,12.0,2012.0,533.0,...,1,0,1,2,12.0,12.0,12,24,20,1
2,5,0.0,0.0,1.0,2.0,12407.0,0.0,11.0,555.0,101.0,...,0,0,0,1,8.0,8.0,8,8,8,1
3,7,0.0,0.0,2.0,1.0,7493.0,1105.0,18.0,84.0,757.0,...,0,0,0,1,4.0,4.0,4,4,4,1
4,8,0.0,0.0,3.0,1.0,9187.0,10841.0,17.0,380.0,592.0,...,0,0,0,0,0.0,0.0,0,0,0,0


## Row Preservation Checks

In [6]:
assert len(feature_table) == len(customer_info), "Feature table row count changed."
assert feature_table["customer_id"].is_unique, "customer_id is not unique after feature engineering."
assert set(feature_table["customer_id"]) == set(customer_info["customer_id"]), "Customer IDs do not match customer_info."
assert metadata["basket_parse_errors"] == 0, "Some basket rows could not be parsed."

no_basket = feature_table["basket_count"] == 0
assert (feature_table.loc[no_basket, "avg_basket_size"] == 0).all(), "No-basket customers must have avg_basket_size = 0."
assert (feature_table.loc[no_basket, "unique_basket_products"] == 0).all(), "No-basket customers must have unique_basket_products = 0."

print("PASS: every customer from customer_info is present exactly once.")
print(f"Customers before: {customers_before:,}")
print(f"Customers after: {customers_after:,}")
print(f"Customers without baskets retained: {int(no_basket.sum()):,}")

PASS: every customer from customer_info is present exactly once.
Customers before: 33,038
Customers after: 33,038
Customers without baskets retained: 4,911


## Missing Values After Preprocessing

In [7]:
post_missing = missing_values(feature_table)
display(post_missing if not post_missing.empty else pd.DataFrame({"message": ["No missing values remain after preprocessing."]}))
print(f"Total missing values after preprocessing: {int(feature_table.isna().sum().sum()):,}")

,message
0,No missing values remain after preprocessing.


Total missing values after preprocessing: 0


## Engineered Feature Distributions

In [8]:
selected_features = [
    "customer_age",
    "customer_tenure_years",
    "total_lifetime_spend",
    "promotion_pct_clean",
    "total_children_home",
    "basket_count",
    "avg_basket_size",
    "unique_basket_products",
]

distribution = feature_table[selected_features].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
display(distribution)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
customer_age,33038.0,55.026459,18.022948,24.40794,25.003066,27.339357,39.252567,54.902122,70.661191,83.296783,85.737522,86.406571
customer_tenure_years,33038.0,11.078637,4.506589,0.00000,1.000000,5.000000,8.000000,11.000000,14.000000,19.000000,22.000000,33.000000
total_lifetime_spend,33038.0,23706.236031,13360.280886,2174.00000,5744.110000,8761.700000,14482.000000,20269.000000,29163.000000,50810.600000,69415.040000,116366.000000
promotion_pct_clean,33038.0,0.324560,0.271470,0.00000,0.000000,0.000000,0.123383,0.239449,0.465200,0.934751,1.000000,1.000000
total_children_home,33038.0,2.014862,1.771471,0.00000,0.000000,0.000000,1.000000,2.000000,2.000000,6.000000,9.000000,14.000000
basket_count,33038.0,3.026818,2.854824,0.00000,0.000000,0.000000,1.000000,2.000000,4.000000,8.000000,13.000000,33.000000
avg_basket_size,33038.0,7.973883,4.121901,0.00000,0.000000,0.000000,6.229167,9.000000,10.666667,13.500000,16.000000,18.000000
unique_basket_products,33038.0,23.759035,20.257936,0.00000,0.000000,0.000000,9.000000,20.000000,35.000000,62.000000,87.630000,145.000000


## Key Feature Groups Created

In [9]:
feature_groups = {
    "demographic": ["customer_age", "gender_female", "gender_male", "gender_unknown"],
    "family": ["kids_home", "teens_home", "total_children_home", "has_children_home"],
    "spend": [column for column in feature_table.columns if column.startswith("spend_share_")] + ["total_lifetime_spend"],
    "loyalty_complaints": ["has_loyalty_card", "loyalty_card_missing", "number_complaints"],
    "promotion": ["promotion_pct_clean", "promotion_pct_suspicious", "promotion_pct_missing"],
    "basket": ["basket_count", "avg_basket_size", "unique_basket_products", "has_sampled_basket"],
}

for group, columns in feature_groups.items():
    present = [column for column in columns if column in feature_table.columns]
    print(f"{group}: {len(present)} features")
    print(present)
    print()

demographic: 4 features
['customer_age', 'gender_female', 'gender_male', 'gender_unknown']

family: 4 features
['kids_home', 'teens_home', 'total_children_home', 'has_children_home']

spend: 11 features
['spend_share_groceries', 'spend_share_electronics', 'spend_share_vegetables', 'spend_share_nonalcohol_drinks', 'spend_share_alcohol_drinks', 'spend_share_meat', 'spend_share_fish', 'spend_share_hygiene', 'spend_share_videogames', 'spend_share_petfood', 'total_lifetime_spend']

loyalty_complaints: 3 features
['has_loyalty_card', 'loyalty_card_missing', 'number_complaints']

promotion: 3 features
['promotion_pct_clean', 'promotion_pct_suspicious', 'promotion_pct_missing']

basket: 4 features
['basket_count', 'avg_basket_size', 'unique_basket_products', 'has_sampled_basket']



## Customers With Baskets vs Without Baskets

In [10]:
comparison_features = [
    "customer_age",
    "customer_tenure_years",
    "total_lifetime_spend",
    "promotion_pct_clean",
    "has_loyalty_card",
    "number_complaints",
    "basket_count",
    "avg_basket_size",
    "unique_basket_products",
]

basket_comparison = (
    feature_table.assign(basket_status=feature_table["has_sampled_basket"].map({1: "with baskets", 0: "without baskets"}))
    .groupby("basket_status")[comparison_features]
    .agg(["count", "mean", "median"])
)
display(basket_comparison)

customer_age                       customer_tenure_years  \
                       count       mean     median                 count   
basket_status                                                              
with baskets           28127  55.110852  54.902122                 28127   
without baskets         4911  54.543110  54.433949                  4911   

                                  total_lifetime_spend                         \
                      mean median                count          mean   median   
basket_status                                                                   
with baskets     11.097166   11.0                28127  23735.852135  20317.0   
without baskets  10.972511   11.0                 4911  23536.614335  20002.0   

                promotion_pct_clean  ... number_complaints basket_count  \
                              count  ...            median        count   
basket_status                        ...                                  
with baskets                  28127  ...               1.0        28127   
without baskets                4911  ...               1.0         4911   

                                 avg_basket_size                     \
                     mean median           count     mean    median   
basket_status                                                         
with baskets     3.555303    3.0           28127  9.36613  9.333333   
without baskets  0.000000    0.0            4911  0.00000  0.000000   

                unique_basket_products                    
                                 count       mean median  
basket_status                                             
with baskets                     28127  27.907384   24.0  
without baskets                   4911   0.000000    0.0  

[2 rows x 27 columns]

## Data Quality Flags to Carry Forward

In [11]:
quality_flag_columns = [
    column
    for column in feature_table.columns
    if column.endswith("_was_missing")
    or column.endswith("_missing")
    or column.endswith("_missing_or_invalid")
    or column.endswith("_suspicious")
    or column.endswith("_parse_failed")
]

quality_flags = (
    feature_table[quality_flag_columns]
    .sum()
    .sort_values(ascending=False)
    .rename("flagged_rows")
    .reset_index()
    .rename(columns={"index": "feature"})
)
display(quality_flags[quality_flags["flagged_rows"] > 0])

,feature,flagged_rows
0,loyalty_card_missing,13106
1,promotion_pct_suspicious,1755
2,customer_tenure_years_was_missing,991
3,first_transaction_year_suspicious,991
4,customer_tenure_missing_or_invalid,991
5,lifetime_spend_fish_was_missing,991
6,lifetime_spend_petfood_was_missing,661
7,lifetime_spend_videogames_was_missing,661
8,number_complaints_was_missing,661
9,lifetime_spend_meat_was_missing,661


## Phase Summary

In [12]:
summary = {
    "customers_before": customers_before,
    "customers_after": customers_after,
    "feature_columns_including_customer_id": feature_table.shape[1],
    "customers_without_baskets": int((feature_table["basket_count"] == 0).sum()),
    "total_missing_after_preprocessing": int(feature_table.isna().sum().sum()),
    "clustering_performed": False,
}
display(pd.DataFrame([summary]))
print("Feature engineering complete. No clustering was performed and no final output file was created.")

,customers_before,customers_after,feature_columns_including_customer_id,customers_without_baskets,total_missing_after_preprocessing,clustering_performed
0,33038,33038,77,4911,0,False


Feature engineering complete. No clustering was performed and no final output file was created.
